# FI'L AI — CAMeLBERT-DA fine-tuning (Google Colab)
This notebook trains only on a CSV you explicitly upload. It accepts only `label_status=human_verified` and does not upload data or checkpoints anywhere else. MASSIVE original labels are not FI'L AI labels.

In [ ]:
!pip -q install 'transformers>=4.45,<5' 'datasets>=3.2,<4' 'accelerate>=1,<2' 'scikit-learn>=1.4,<2' pandas
import random, json, shutil, numpy as np, pandas as pd, torch
from google.colab import files
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, EarlyStoppingCallback, Trainer, TrainingArguments
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('CUDA:', torch.cuda.is_available())

## Upload reviewed data
Upload `annotation_template.csv` after human review. Required columns: `message`, `intent`, `label_status`.

In [ ]:
uploaded = files.upload()
csv_name = next(iter(uploaded))
df = pd.read_csv(csv_name)
LABELS=['new_task','reschedule','cancellation','info_only']
required={'message','intent','label_status'}
assert required <= set(df.columns), f'Missing {required-set(df.columns)}'
df = df[df.label_status.eq('human_verified') & df.intent.isin(LABELS)].dropna(subset=['message','intent']).copy()
counts=df.intent.value_counts().reindex(LABELS,fill_value=0)
print(len(df), counts.to_dict())
assert len(df)>=80 and counts.min()>=10, 'Blocked: need >=80 verified rows and >=10/class'

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df.intent, random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df.intent, random_state=SEED)
MODEL_ID='CAMeL-Lab/bert-base-arabic-camelbert-da'
label2id={label:i for i,label in enumerate(LABELS)}; id2label={i:label for label,i in label2id.items()}
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
def make_dataset(frame):
    ds=Dataset.from_pandas(frame[['message','intent']], preserve_index=False)
    def tokenize(batch):
        out=tokenizer(batch['message'], truncation=True, max_length=128)
        out['labels']=[label2id[x] for x in batch['intent']]
        return out
    return ds.map(tokenize, batched=True)
train_ds,val_ds,test_ds=map(make_dataset,[train_df,val_df,test_df])

In [ ]:
model=AutoModelForSequenceClassification.from_pretrained(MODEL_ID,num_labels=4,label2id=label2id,id2label=id2label)
def compute_metrics(result):
    pred=np.argmax(result.predictions,axis=-1); true=result.label_ids
    return {'accuracy':accuracy_score(true,pred),'macro_f1':f1_score(true,pred,average='macro',zero_division=0),'weighted_f1':f1_score(true,pred,average='weighted',zero_division=0)}
args=TrainingArguments(output_dir='camelbert_runs',learning_rate=2e-5,per_device_train_batch_size=8,per_device_eval_batch_size=8,gradient_accumulation_steps=2,num_train_epochs=4,eval_strategy='epoch',save_strategy='epoch',load_best_model_at_end=True,metric_for_best_model='eval_macro_f1',greater_is_better=True,seed=SEED,report_to=[])
trainer=Trainer(model=model,args=args,train_dataset=train_ds,eval_dataset=val_ds,processing_class=tokenizer,data_collator=DataCollatorWithPadding(tokenizer),compute_metrics=compute_metrics,callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.train()

In [ ]:
test_output=trainer.predict(test_ds)
test_pred=np.argmax(test_output.predictions,axis=-1)
print(test_output.metrics)
print(classification_report(test_output.label_ids,test_pred,target_names=LABELS,zero_division=0))
trainer.save_model('camelbert_fil_ai'); tokenizer.save_pretrained('camelbert_fil_ai')
with open('camelbert_fil_ai/evaluation.json','w') as f: json.dump(test_output.metrics,f,indent=2)
shutil.make_archive('camelbert_fil_ai','zip','camelbert_fil_ai')
files.download('camelbert_fil_ai.zip')

## Local loading
Unzip into `models/camelbert_fil_ai/`, then use `AutoTokenizer.from_pretrained(path)` and `AutoModelForSequenceClassification.from_pretrained(path)`. Keep the test split and `evaluation.json` with the checkpoint. Do not compare these metrics to another test set.